# Hamiltonian Neural Networks — Learning Conservation Laws from Data

**System:** Nonlinear pendulum  $H(q,p) = p^2/2 + (1-\cos q)$

**Goal:** Train a neural network that learns the Hamiltonian $H$ from noisy trajectory
data, then derive dynamics via $\dot{q} = \partial H/\partial p$, $\dot{p} = -\partial H/\partial q$.
Compare long-horizon rollouts against a standard MLP that learns derivatives directly.

---

## Companion to Notebook 01

| Notebook 01 | Notebook 02 |
|---|---|
| HMC uses Hamiltonian dynamics to *sample* a posterior | HNN *learns* the Hamiltonian from data |
| Physics → Statistics (sampling) | Data → Physics (structure identification) |
| leapfrog integrates known $H$ | network approximates unknown $H$ |

The key result: **a network constrained to output an energy function automatically
produces energy-conserving rollouts** — the conservation law is baked into the
architecture, not learned from data.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm

from src.hnn import MLP, train_hnn, train_baseline
from src.pendulum import generate_data, rollout, pendulum_H, pendulum_derivatives

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12})


## 2. Data: Nonlinear Pendulum Trajectories

The pendulum has one degree of freedom $(q, p)$ — angle and angular momentum —
with Hamiltonian:

$$H(q, p) = \frac{p^2}{2} + (1 - \cos q)$$

We integrate 40 trajectories from random initial conditions, sample 50 points each,
and add small Gaussian noise ($\sigma = 0.01$) to simulate realistic measurements.

**What both models see during training:**
- Baseline MLP: $(q, p) \to (\dot{q}, \dot{p})$ pairs
- HNN: $(q, p) \to H$ pairs  (energy labels — the conserved quantity)

Neither model sees trajectories at test time; they must generalise from point data.


In [ ]:
data = generate_data(n_traj=40, n_points=50, noise_std=0.01, seed=42)

X      = data['X_noisy']   # (N, 2) noisy (q, p) — training input
dX     = data['dX']        # (N, 2) true derivatives
H_true = data['H']         # (N,)  true energy values

print(f"Training points : {len(X)}")
print(f"H range         : [{H_true.min():.3f}, {H_true.max():.3f}]")
print(f"q range         : [{X[:,0].min():.3f}, {X[:,0].max():.3f}]")
print(f"p range         : [{X[:,1].min():.3f}, {X[:,1].max():.3f}]")


In [ ]:
# ── Phase portrait: training data coloured by energy ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc = axes[0].scatter(X[:,0], X[:,1], c=H_true, cmap='plasma', s=8, alpha=0.7)
plt.colorbar(sc, ax=axes[0], label='H(q, p)')
axes[0].set_xlabel('q  (angle)')
axes[0].set_ylabel('p  (angular momentum)')
axes[0].set_title('Training Data — Phase Portrait', fontweight='bold')

# True Hamiltonian surface
q_grid = np.linspace(-3.2, 3.2, 120)
p_grid = np.linspace(-2.5, 2.5, 100)
Q, P = np.meshgrid(q_grid, p_grid)
H_grid = pendulum_H(Q, P)

cf = axes[1].contourf(Q, P, H_grid, levels=20, cmap='plasma', alpha=0.85)
axes[1].contour(Q, P, H_grid, levels=[1.0, 2.0], colors='white', linewidths=1.2,
               linestyles='--', alpha=0.7)
plt.colorbar(cf, ax=axes[1], label='True H(q, p)')
axes[1].set_xlabel('q'); axes[1].set_ylabel('p')
axes[1].set_title('True Hamiltonian Surface', fontweight='bold')

fig.suptitle('Pendulum Phase Space', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 3. Two Models

### Baseline MLP
Standard regression: learn $(\dot{q}, \dot{p})$ from $(q, p)$.

$$\mathcal{L}_{\text{baseline}} = \frac{1}{N}\sum_i \left[(\dot{q}_i^{\text{pred}} - \dot{q}_i)^2 + (\dot{p}_i^{\text{pred}} - \dot{p}_i)^2\right]$$

### Hamiltonian Neural Network
Learn $H(q,p)$ from energy labels. Dynamics come from the **analytic Jacobian**
$\partial H_{\text{net}} / \partial (q, p)$:

$$\mathcal{L}_{\text{HNN}} = \frac{1}{N}\sum_i (H_{\text{net}}(q_i, p_i) - H_i)^2$$

$$\dot{q} = \frac{\partial H_{\text{net}}}{\partial p}, \qquad \dot{p} = -\frac{\partial H_{\text{net}}}{\partial q}$$

The Jacobian is computed analytically via the chain rule:

$$\frac{\partial H}{\partial x} = W_1^\top \odot (1-\tanh^2 z_1) \cdot W_2^\top \odot (1-\tanh^2 z_2) \cdot W_3$$

No automatic differentiation is needed — the formula is closed-form for a
2-hidden-layer MLP.


## 4. Training

In [ ]:
print("Training Baseline MLP (learns derivatives directly) ...")
baseline = MLP(n_in=2, n_out=2, hidden=128, seed=0)
loss_baseline = train_baseline(baseline, X, dX,
                               n_epochs=4000, batch_size=256, lr=1e-3,
                               seed=0, verbose=True)

print("\nTraining Hamiltonian Neural Network (learns H) ...")
hnn = MLP(n_in=2, n_out=1, hidden=128, seed=1)
loss_hnn = train_hnn(hnn, X, H_true,
                     n_epochs=4000, batch_size=256, lr=1e-3,
                     seed=1, verbose=True)


In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogy(loss_baseline, color='#bf3a3a', lw=1.5)
axes[0].set_title('Baseline MLP — MSE(derivatives)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log scale)')

axes[1].semilogy(loss_hnn, color='#3a7abf', lw=1.5)
axes[1].set_title('HNN — MSE(H)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss (log scale)')

fig.suptitle('Training Convergence', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 5. What the HNN Learned

We visualise the learned Hamiltonian surface $H_{\text{net}}(q, p)$ against
the true $H(q, p) = p^2/2 + (1 - \cos q)$.

A well-trained HNN should recover the correct contour topology — **the level sets
are the conserved-energy orbits**, so getting the topology right is equivalent
to identifying the correct conservation law.


In [ ]:
# ── Learned vs true Hamiltonian surface ───────────────────────────────────────
q_grid = np.linspace(-3.0, 3.0, 100)
p_grid = np.linspace(-2.4, 2.4, 100)
Q, P = np.meshgrid(q_grid, p_grid)
XY = np.column_stack([Q.ravel(), P.ravel()])

H_pred_grid = hnn.forward(XY).reshape(Q.shape)
H_true_grid = pendulum_H(Q, P)

# HNN output is defined up to an additive constant — align means
H_pred_grid -= H_pred_grid.mean()
H_pred_grid += H_true_grid.mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

vmin = H_true_grid.min(); vmax = min(H_true_grid.max(), 3.0)

for ax, Z, title in [
    (axes[0], H_true_grid, 'True  H(q, p)'),
    (axes[1], H_pred_grid, 'Learned  $H_{net}$(q, p)'),
    (axes[2], H_pred_grid - H_true_grid, 'Error  $H_{net} - H$'),
]:
    if 'Error' in title:
        cf = ax.contourf(Q, P, Z, levels=20, cmap='RdBu_r', alpha=0.9)
    else:
        cf = ax.contourf(Q, P, Z, levels=20, cmap='plasma',
                         vmin=vmin, vmax=vmax, alpha=0.9)
        ax.contour(Q, P, Z, levels=8, colors='white', linewidths=0.7, alpha=0.5)
    plt.colorbar(cf, ax=ax)
    ax.set_xlabel('q'); ax.set_ylabel('p')
    ax.set_title(title, fontweight='bold')
    ax.scatter(X[:,0], X[:,1], s=3, c='white', alpha=0.25, zorder=3)

fig.suptitle('Learned vs True Hamiltonian', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Long-Horizon Rollouts — Energy Conservation

Starting from an **unseen initial condition**, we integrate both models forward
for 500 steps using RK4 and track:

1. Phase space trajectory — should trace a closed orbit for a conservative system
2. Energy $H(q(t), p(t))$ over time — should be constant

The baseline MLP has no physics constraint, so energy drifts.
The HNN derives dynamics from $\partial H_{\text{net}}$, so energy is
approximately conserved by construction.


In [ ]:
# ── Define dynamics functions for each model ──────────────────────────────────
def baseline_dyn(q, p):
    x = np.array([[q, p]])
    dx = baseline.forward(x)[0]
    return float(dx[0]), float(dx[1])

def hnn_dyn(q, p):
    x = np.array([[q, p]])
    J = hnn.input_jacobian(x)[0]   # ∂H/∂(q,p) — shape (2,)
    return float(J[1]), float(-J[0])  # dq/dt = ∂H/∂p,  dp/dt = -∂H/∂q

# Test initial condition (not in training data)
q0, p0 = 1.2, 0.8
n_steps = 500
dt = 0.1

qs_true,     ps_true     = rollout(q0, p0, n_steps, dt, pendulum_derivatives)
qs_baseline, ps_baseline = rollout(q0, p0, n_steps, dt, baseline_dyn)
qs_hnn,      ps_hnn      = rollout(q0, p0, n_steps, dt, hnn_dyn)

H_true_traj     = pendulum_H(qs_true,     ps_true)
H_baseline_traj = pendulum_H(qs_baseline, ps_baseline)
H_hnn_traj      = pendulum_H(qs_hnn,      ps_hnn)

t = np.arange(n_steps + 1) * dt

print(f"Initial energy H₀ = {pendulum_H(q0, p0):.4f}")
print(f"True      ΔH = {np.abs(H_true_traj - H_true_traj[0]).max():.2e}")
print(f"Baseline  ΔH = {np.abs(H_baseline_traj - H_baseline_traj[0]).max():.4f}")
print(f"HNN       ΔH = {np.abs(H_hnn_traj - H_hnn_traj[0]).max():.4f}")


In [ ]:
# ── Comparison plots ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

colors = {'true': '#333333', 'baseline': '#bf3a3a', 'hnn': '#3a7abf'}

# Row 0: Phase portraits
for col, (qs, ps, label, c) in enumerate([
    (qs_true,     ps_true,     'True (scipy RK45)',  colors['true']),
    (qs_baseline, ps_baseline, 'Baseline MLP',       colors['baseline']),
    (qs_hnn,      ps_hnn,      'HNN',                colors['hnn']),
]):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(qs, ps, lw=1.2, color=c, alpha=0.85)
    ax.plot(q0, p0, 'o', ms=8, color='green', zorder=5)
    ax.set_xlabel('q  (angle)'); ax.set_ylabel('p')
    ax.set_title(f'Phase Space — {label}', fontweight='bold', fontsize=10)

# Row 1: Energy over time
ax_E = fig.add_subplot(gs[1, :])
ax_E.plot(t, H_true_traj,     lw=2.0, color=colors['true'],     label='True (RK45)',   alpha=0.9)
ax_E.plot(t, H_baseline_traj, lw=1.5, color=colors['baseline'], label='Baseline MLP',  alpha=0.9, ls='--')
ax_E.plot(t, H_hnn_traj,      lw=1.5, color=colors['hnn'],      label='HNN',           alpha=0.9, ls='-.')
ax_E.axhline(pendulum_H(q0, p0), color='grey', lw=0.8, ls=':', alpha=0.7, label='H₀')
ax_E.set_xlabel('Time  t'); ax_E.set_ylabel('H(q(t), p(t))')
ax_E.set_title('Energy Conservation Over Time', fontweight='bold')
ax_E.legend(fontsize=10)

fig.suptitle('Long-Horizon Rollout Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.show()


## 7. Derivative Quality at Training Points

The HNN was never trained on derivative labels — it saw only energy values.
Yet its dynamics $\dot{q} = \partial H_{\text{net}}/\partial p$,
$\dot{p} = -\partial H_{\text{net}}/\partial q$ implicitly predict derivatives.

We compare both models' derivative predictions on the training data.


In [ ]:
# Baseline predictions
dX_baseline = baseline.forward(X)      # (N, 2)

# HNN derivative predictions via Jacobian
J = hnn.input_jacobian(X)              # (N, 2) — [∂H/∂q, ∂H/∂p]
dX_hnn = np.column_stack([J[:, 1], -J[:, 0]])  # [∂H/∂p, -∂H/∂q]

mse_baseline = np.mean((dX_baseline - dX) ** 2)
mse_hnn      = np.mean((dX_hnn      - dX) ** 2)
print(f"Derivative MSE — Baseline MLP : {mse_baseline:.6f}  (trained on this)")
print(f"Derivative MSE — HNN          : {mse_hnn:.6f}  (zero-shot from Jacobian)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
subset = slice(None, 300)

for col, (model_dx, title, c) in enumerate([
    (dX_baseline, 'Baseline MLP — dq/dt', colors['baseline']),
    (dX_hnn,      'HNN (∂H/∂p) — dq/dt', colors['hnn']),
]):
    ax = axes[col]
    ax.scatter(dX[subset, 0], model_dx[subset, 0], s=8, alpha=0.5, color=c)
    lims = [min(dX[:,0].min(), model_dx[:,0].min()),
            max(dX[:,0].max(), model_dx[:,0].max())]
    ax.plot(lims, lims, 'k--', lw=1.2, alpha=0.6, label='Perfect')
    ax.set_xlabel('True dq/dt'); ax.set_ylabel('Predicted dq/dt')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Derivative Prediction Quality (first 300 points)', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 8. The Learned Force Field

The HNN Jacobian $\nabla_x H_{\text{net}}(q, p)$ defines a vector field
over phase space. Since $\dot{q} = \partial H/\partial p$ and $\dot{p} = -\partial H/\partial q$,
this is the **force field** the model uses for rollouts.

A well-trained HNN should show a force field that rotates clockwise (for the pendulum)
and aligns with the true level sets of $H$.


In [ ]:
# ── Force field vector plot ────────────────────────────────────────────────────
q_vals = np.linspace(-2.8, 2.8, 18)
p_vals = np.linspace(-2.2, 2.2, 14)
QQ, PP = np.meshgrid(q_vals, p_vals)
XY_grid = np.column_stack([QQ.ravel(), PP.ravel()])

J_grid = hnn.input_jacobian(XY_grid)
dq_hnn = J_grid[:, 1].reshape(QQ.shape)    # ∂H/∂p
dp_hnn = -J_grid[:, 0].reshape(QQ.shape)   # -∂H/∂q

dq_true = PP.copy()                          # true: dq/dt = p
dp_true = -np.sin(QQ)                        # true: dp/dt = -sin(q)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, dq, dp, title, c in [
    (axes[0], dq_true, dp_true, 'True Force Field', colors['true']),
    (axes[1], dq_hnn,  dp_hnn,  'HNN Force Field  (∂H_net/∂x)', colors['hnn']),
]:
    speed = np.sqrt(dq**2 + dp**2)
    ax.streamplot(q_vals, p_vals, dq, dp,
                  color=speed, cmap='Blues', linewidth=1.2, density=1.2, arrowsize=1.1)
    ax.set_xlim(-2.8, 2.8); ax.set_ylim(-2.2, 2.2)
    ax.set_xlabel('q'); ax.set_ylabel('p')
    ax.set_title(title, fontweight='bold')

fig.suptitle('Phase Space Force Fields', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 9. Summary

| Property | Baseline MLP | HNN |
|---|---|---|
| **Training target** | $(\dot{q}, \dot{p})$ derivatives | $H(q, p)$ energy values |
| **Architecture output** | 2D vector | Scalar $H$ |
| **Dynamics source** | Direct network output | Analytic Jacobian $\partial H_{\text{net}}/\partial x$ |
| **Energy conservation** | No — drifts on rollouts | Approximate — baked into structure |
| **Phase space topology** | Depends on data density | Correct by construction near training data |
| **Data requirement** | Derivative labels | Energy labels |

### Key takeaway

The Hamiltonian is not merely a function the HNN learns — it is the **inductive bias**
that encodes conservation laws into the model. By restricting the network to
output a scalar potential and deriving dynamics via Hamilton's equations, we
guarantee that any conserved quantity of the true Hamiltonian is approximately
reproduced, even on trajectories the model has never seen.

This is the same principle that powers structure-preserving integrators in
classical mechanics (the leapfrog in Notebook 01) — but now applied to
*learning* the structure from data rather than exploiting a known one.

### Natural extensions

- **Symplectic integrator for HNN rollouts** — replace RK4 with leapfrog on $H_{\text{net}}$
  for exact discrete energy conservation
- **Lagrangian Neural Networks** (Cranmer et al. 2020) — learn $L(q, \dot{q})$ instead
  of $H(q, p)$; equivalent but works in configuration space
- **Port-Hamiltonian networks** — extend to dissipative systems with input/output ports
- **Neural ODEs** — parameterise the full vector field as an ODE and learn it end-to-end
